In [85]:
import pandas as pd

df_sch_info = pd.read_csv('General information of schools.csv')
df_subjects = pd.read_csv('Subjects Offered.csv')
df_distprogrammes = pd.read_csv('School Distinctive Programmes.csv')
df_ccas = pd.read_csv('Co-curricular activities (CCAs).csv')

In [86]:
import openpyxl

df_affiliations = pd.read_excel('school_affiliations_psle.xlsx', sheet_name=0)
df_psle = pd.read_excel('school_affiliations_psle.xlsx', sheet_name=1)

print(df_affiliations.head())
print(df_psle.head())

                      school_name                   affiliated_school
0   Anglo-Chinese School (Junior)  Anglo-Chinese School (Barker Road)
1   Anglo-Chinese School (Junior)  Anglo-Chinese School (Independent)
2  Anglo-Chinese School (Primary)  Anglo-Chinese School (Barker Road)
3  Anglo-Chinese School (Primary)  Anglo-Chinese School (Independent)
4            Catholic High School                Catholic High School
   rank                     school_name median_psle_score_range
0     1   Raffles Girls' Primary School                     4-7
1     2          Nanyang Primary School                     4-7
2     3            Catholic High School                     5-8
3     4  Anglo-Chinese School (Primary)                     5-8
4     5       Henry Park Primary School                     5-8


In [115]:
import requests
from bs4 import BeautifulSoup

url = "https://www.property2b2c.com/school-ranking/2025-pop"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

tables = pd.read_html(str(soup.find_all("table")))
df_ballot = tables[0]

print(df_ballot)



/var/folders/2s/sj7n8qbs5838w7lml1xsqlv40000gn/T/ipykernel_4396/2403609962.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup.find_all("table")))


    No.                School  Unnamed: 2  GEP  SAP Gender           Area  \
0     1               Tao Nan         NaN  GEP  SAP    NaN  Marine Parade   
1     2               Ai Tong         NaN  NaN  SAP    NaN         Bishan   
2     3               Nanyang         NaN  GEP  SAP    NaN    Bukit Timah   
3     4  Pei Hwa Presbyterian         NaN  NaN  SAP    NaN    Bukit Timah   
4     5      Methodist Girls'         NaN  NaN  NaN   Girl    Bukit Timah   
..   ..                   ...         ...  ...  ...    ...            ...   
174   -                Yishun         NaN  NaN  NaN    NaN         Yishun   
175   -                 Yumin         NaN  NaN  NaN    NaN       Tampines   
176   -               Zhangde         NaN  NaN  NaN    NaN    Bukit Merah   
177   -              Zhenghua         NaN  NaN  NaN    NaN  Bukit Panjang   
178   -              Zhonghua         NaN  NaN  NaN    NaN      Serangoon   

     Vacancy  Applicant Admission Popularity  
0         20         63     

In [ ]:
# Create a mapping function to match schools based on first few words
def fuzzy_match_schools(ballot_school, sch_info_names, num_words):
    ballot_words = ballot_school.upper().split()[:num_words]
    ballot_prefix = ' '.join(ballot_words)
    
    for sch_name in sch_info_names:
        sch_words = sch_name.upper().split()[:num_words]
        sch_prefix = ' '.join(sch_words)
        
        if ballot_prefix == sch_prefix:
            return sch_name
    
    return None

# Apply fuzzy matching to map df_ballot schools to df_sch_info
df_ballot['school_name_matched'] = df_ballot['School'].apply(
    lambda x: fuzzy_match_schools(x, df_sch_info['school_name'].values, num_words=1 or 2 or 3)
)

print(df_ballot[['School', 'school_name_matched']])

                   School                school_name_matched
0                 Tao Nan                     TAO NAN SCHOOL
1                 Ai Tong                     AI TONG SCHOOL
2                 Nanyang             NANYANG PRIMARY SCHOOL
3    Pei Hwa Presbyterian             PEI CHUN PUBLIC SCHOOL
4        Methodist Girls'  METHODIST GIRLS' SCHOOL (PRIMARY)
..                    ...                                ...
174                Yishun              YISHUN PRIMARY SCHOOL
175                 Yumin               YUMIN PRIMARY SCHOOL
176               Zhangde             ZHANGDE PRIMARY SCHOOL
177              Zhenghua            ZHENGHUA PRIMARY SCHOOL
178              Zhonghua            ZHONGHUA PRIMARY SCHOOL

[179 rows x 2 columns]


In [88]:
df_sch_info = df_sch_info[df_sch_info['mainlevel_code'].isin(['PRIMARY', 'MIXED LEVEL (P1-S4)'])]
df_sch_info = df_sch_info[['school_name', 'address', 'postal_code', 'nature_code', 'sap_ind', 'autonomous_ind', 'gifted_ind']]
df_sch_info = df_sch_info.replace({'na': 0, 'Yes': 1, 'No': 0})
print(df_sch_info)

                      school_name                        address  postal_code  \
0        ADMIRALTY PRIMARY SCHOOL         11 WOODLANDS CIRCLE          738907   
2    AHMAD IBRAHIM PRIMARY SCHOOL         10 YISHUN STREET 11          768643   
4                  AI TONG SCHOOL       100 Bright Hill Drive          579646   
5        ALEXANDRA PRIMARY SCHOOL  2A Prince Charles Crescent          159016   
6     ANCHOR GREEN PRIMARY SCHOOL         31 Anchorvale Drive          544969   
..                            ...                            ...          ...   
327          YUHUA PRIMARY SCHOOL   158 JURONG EAST STREET 24          609558   
329          YUMIN PRIMARY SCHOOL        3 TAMPINES STREET 21          529393   
332        ZHANGDE PRIMARY SCHOOL            51 Jalan Membina          169485   
333       ZHENGHUA PRIMARY SCHOOL                9 Fajar Road          679002   
335       ZHONGHUA PRIMARY SCHOOL       12 SERANGOON AVENUE 4          556095   

      nature_code  sap_ind 

/var/folders/2s/sj7n8qbs5838w7lml1xsqlv40000gn/T/ipykernel_4396/3655042808.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sch_info = df_sch_info.replace({'na': 0, 'Yes': 1, 'No': 0})


In [89]:
# Add 'school' to school_name if it doesn't end with 'school'
df_distprogrammes['school_name'] = df_distprogrammes['school_name'].apply(
    lambda x: x + ' school' if not x.lower().endswith('school') else x
)
df_distprogrammes['school_name'] = df_distprogrammes['school_name'].str.upper()

In [90]:
# Filter to include only schools from df_sch_info
df_subjects = df_subjects[df_subjects['School_Name'].isin(df_sch_info['school_name'])]
df_ccas = df_ccas[df_ccas['School_name'].isin(df_sch_info['school_name'])]
df_distprogrammes = df_distprogrammes[df_distprogrammes['school_name'].isin(df_sch_info['school_name'])]

print(df_ccas.head())
print(df_subjects.head())
print(df_distprogrammes.head())

                School_name school_section  \
0  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
1  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
2  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
3  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
4  ADMIRALTY PRIMARY SCHOOL        PRIMARY   

                      cca_grouping_desc            cca_generic_name  \
0                        ART AND CRAFTS         CLUBS AND SOCIETIES   
1                         CHINESE DANCE  VISUAL AND PERFORMING ARTS   
2                                 CHOIR  VISUAL AND PERFORMING ARTS   
3                 DESIGN AND INNOVATION         CLUBS AND SOCIETIES   
4  ENGLISH LANGUAGE, DRAMA AND DEBATING         CLUBS AND SOCIETIES   

  cca_customized_name  
0                  na  
1                  na  
2                  na  
3                  na  
4                  na  
                School_Name                 Subject_Desc
0  ADMIRALTY PRIMARY SCHOOL                          Art
1  ADMIRALTY PRIMARY SCHOOL           

In [91]:
# Keep only relevant columns for df_ccas
df_ccas = df_ccas[['School_name', 'cca_grouping_desc', 'cca_generic_name']]

# Keep only relevant columns for df_distprogrammes
df_distprogrammes = df_distprogrammes[['school_name', 'alp_domain']]

print(df_ccas.head())
print(df_distprogrammes.head())

                School_name                     cca_grouping_desc  \
0  ADMIRALTY PRIMARY SCHOOL                        ART AND CRAFTS   
1  ADMIRALTY PRIMARY SCHOOL                         CHINESE DANCE   
2  ADMIRALTY PRIMARY SCHOOL                                 CHOIR   
3  ADMIRALTY PRIMARY SCHOOL                 DESIGN AND INNOVATION   
4  ADMIRALTY PRIMARY SCHOOL  ENGLISH LANGUAGE, DRAMA AND DEBATING   

             cca_generic_name  
0         CLUBS AND SOCIETIES  
1  VISUAL AND PERFORMING ARTS  
2  VISUAL AND PERFORMING ARTS  
3         CLUBS AND SOCIETIES  
4         CLUBS AND SOCIETIES  
                      school_name                alp_domain
103  AHMAD IBRAHIM PRIMARY SCHOOL     STEM - Sustainability
104                AI TONG SCHOOL   Innovation & Enterprise
105      ALEXANDRA PRIMARY SCHOOL   STEM - Material Science
106   ANCHOR GREEN PRIMARY SCHOOL        Interdisciplinary 
107       ANDERSON PRIMARY SCHOOL  STEM  - Material Science


In [96]:
# Pivot df_ccas to create a counter of each cca_generic_name per school
df_ccas_pivot = df_ccas.groupby(['School_name', 'cca_generic_name']).size().unstack(fill_value=0)
df_ccas_pivot = df_ccas_pivot.reset_index()
df_ccas_pivot.columns = ['School_name', 'cca_clubs', 'cca_others', 'cca_sports', 'cca_uniformed', 'cca_arts']
print(df_ccas_pivot.head())

                    School_name  cca_clubs  cca_others  cca_sports  \
0      ADMIRALTY PRIMARY SCHOOL          5           0           5   
1  AHMAD IBRAHIM PRIMARY SCHOOL          3           1           4   
2                AI TONG SCHOOL          4           0           6   
3      ALEXANDRA PRIMARY SCHOOL          6           0           7   
4   ANCHOR GREEN PRIMARY SCHOOL          4           0           4   

   cca_uniformed  cca_arts  
0              2         6  
1              1         4  
2              2         5  
3              1         5  
4              1         5  
                      School_name  cca_clubs  cca_others  cca_sports  \
0        ADMIRALTY PRIMARY SCHOOL          5           0           5   
1    AHMAD IBRAHIM PRIMARY SCHOOL          3           1           4   
2                  AI TONG SCHOOL          4           0           6   
3        ALEXANDRA PRIMARY SCHOOL          6           0           7   
4     ANCHOR GREEN PRIMARY SCHOOL          4 

In [103]:
# Categorize alp_domain into distprog categories
def categorize_alp(domain):
    if 'STEM' in domain.upper():
        return 'STEM'
    elif 'ICT' in domain.upper():
        return 'ICT'
    elif 'INTERDISCIPLINARY' in domain.upper():
        return 'Interdisciplinary'
    elif 'LANGUAGES' in domain.upper():
        return 'Languages'

df_distprogrammes['alp_category'] = df_distprogrammes['alp_domain'].apply(categorize_alp)

# Pivot to count each category per school
df_distprogrammes_pivot = df_distprogrammes.groupby(['school_name', 'alp_category']).size().unstack(fill_value=0)
df_distprogrammes_pivot = df_distprogrammes_pivot.reset_index()
df_distprogrammes_pivot = df_distprogrammes_pivot.rename(columns={
    'STEM': 'distprog_STEM',
    'ICT': 'distprog_ICT',
    'Interdisciplinary': 'distprog_Interdisciplinary',
    'Languages': 'distprog_Languages'
})

print(df_distprogrammes_pivot)

alp_category                   school_name  distprog_ICT  \
0             AHMAD IBRAHIM PRIMARY SCHOOL             0   
1                 ALEXANDRA PRIMARY SCHOOL             0   
2              ANCHOR GREEN PRIMARY SCHOOL             0   
3                  ANDERSON PRIMARY SCHOOL             0   
4                ANG MO KIO PRIMARY SCHOOL             0   
..                                     ...           ...   
122                   YUHUA PRIMARY SCHOOL             0   
123                   YUMIN PRIMARY SCHOOL             0   
124                 ZHANGDE PRIMARY SCHOOL             0   
125                ZHENGHUA PRIMARY SCHOOL             0   
126                ZHONGHUA PRIMARY SCHOOL             0   

alp_category  distprog_Interdisciplinary  distprog_Languages  distprog_STEM  
0                                      0                   0              1  
1                                      0                   0              1  
2                                      1     

In [107]:
# Pivot df_subjects to create a counter of each subject per school
df_subjects_pivot = df_subjects.groupby('School_Name').size().reset_index(name='subject_count')

print(df_subjects_pivot)

                      School_Name  subject_count
0        ADMIRALTY PRIMARY SCHOOL             24
1    AHMAD IBRAHIM PRIMARY SCHOOL             21
2                  AI TONG SCHOOL             14
3        ALEXANDRA PRIMARY SCHOOL             22
4     ANCHOR GREEN PRIMARY SCHOOL             30
..                            ...            ...
176          YUHUA PRIMARY SCHOOL             23
177          YUMIN PRIMARY SCHOOL             23
178        ZHANGDE PRIMARY SCHOOL             29
179       ZHENGHUA PRIMARY SCHOOL             25
180       ZHONGHUA PRIMARY SCHOOL             23

[181 rows x 2 columns]


In [109]:
# Pivot df_affiliations to create a counter of affiliations per school
df_affliation_pivot = df_affiliations.groupby('school_name').size().reset_index(name='affliation_count')

print(df_affliation_pivot)

                                     school_name  affliation_count
0                  Anglo-Chinese School (Junior)                 2
1                 Anglo-Chinese School (Primary)                 2
2                          CHIJ (Katong) Primary                 1
3                                 CHIJ (Kellock)                 1
4                   CHIJ Our Lady Queen of Peace                 1
5                  CHIJ Our Lady of Good Counsel                 1
6                  CHIJ Our Lady of the Nativity                 1
7                       CHIJ Primary (Toa Payoh)                 1
8                CHIJ St. Nicholas Girls' School                 1
9                 Canossa Convent Primary School                 1
10                          Catholic High School                 1
11                            De La Salle School                 2
12          Fairfield Methodist School (Primary)                 1
13            Geylang Methodist School (Primary)              

In [113]:
# Merge all datasets with df_sch_info as the base
df_schools = df_sch_info.copy()

# Merge df_ballot
df_schools = df_schools.merge(df_ballot, left_on='school_name', right_on='School', how='left')

# Merge df_subjects_pivot
df_schools = df_schools.merge(df_subjects_pivot, left_on='school_name', right_on='School_Name', how='left')

# Merge df_distprogrammes_pivot
df_schools = df_schools.merge(df_distprogrammes_pivot, left_on='school_name', right_on='school_name', how='left')

# Merge df_ccas_pivot
df_schools = df_schools.merge(df_ccas_pivot, left_on='school_name', right_on='School_name', how='left')

# Merge df_affliation_pivot
df_schools = df_schools.merge(df_affliation_pivot, left_on='school_name', right_on='school_name', how='left')

# Add top_psle_score column
df_schools['top_psle_score'] = df_schools['school_name'].isin(df_psle['school_name']).astype(int)

# Drop duplicate columns if any
df_schools = df_schools.loc[:, ~df_schools.columns.duplicated()]

print(df_schools.head())
print(df_schools.shape)

                    school_name                        address  postal_code  \
0      ADMIRALTY PRIMARY SCHOOL         11 WOODLANDS CIRCLE          738907   
1  AHMAD IBRAHIM PRIMARY SCHOOL         10 YISHUN STREET 11          768643   
2                AI TONG SCHOOL       100 Bright Hill Drive          579646   
3      ALEXANDRA PRIMARY SCHOOL  2A Prince Charles Crescent          159016   
4   ANCHOR GREEN PRIMARY SCHOOL         31 Anchorvale Drive          544969   

    nature_code  sap_ind  autonomous_ind  gifted_ind  No. School  Unnamed: 2  \
0  CO-ED SCHOOL        0               0           0  NaN    NaN         NaN   
1  CO-ED SCHOOL        0               0           0  NaN    NaN         NaN   
2  CO-ED SCHOOL        1               0           0  NaN    NaN         NaN   
3  CO-ED SCHOOL        0               0           0  NaN    NaN         NaN   
4  CO-ED SCHOOL        0               0           0  NaN    NaN         NaN   

   GEP  SAP Gender Area  Vacancy  Applicant 